# Downscaling validation for air and dew point temperature

In [ ]:
import datetime as dt
import os

import pandas as pd


In [ ]:
from etdataset import era5, icos
from etdataset.api import (
    generate_lapse_rate_for_stations_multiprocess,
    generate_timeseries_for_stations_multiprocess,
)
from etdataset.plot import plot_lr_csv, plot_ta_tdp_csv

In [ ]:
from etdataset.logging import LoggerManager

logger = LoggerManager.get_logger(__name__)


## ICOS stations

Get a csv file with all the valid ICOS stations and their metadata

In [ ]:
path_icos_stat = icos.get_csv_with_valid_icos_stations()


In [ ]:
path_icos_stat = "stations_meteoL2.csv"

Check the available stations by country number

In [ ]:
icos.filter_stations_by_country_code(path_icos_stat, "FR")

In [ ]:
# API token from your ICOS account
auth_token = os.environ["ICOS_API_TOKEN"]

In [ ]:
icos.download_icos_station(auth_token, ["DE-Geb", "FR-CLt"])

In [ ]:
cfg = icos.load_stations_config(path_icos_stat)

## Retrieve the time series for ICOS, ERA5, and downscaled ERA5 datasets.

In [ ]:
output = "out"
os.makedirs(output, exist_ok=True)

In [ ]:
start_date = dt.date(2023, 11, 20)
end_date = dt.date(2023, 12, 20)

Download ERA5 datasets date by date

In [ ]:
era5.download_date_by_date(
    date1=start_date,
    date2=end_date,
    dataset=era5.ERA5Dataset.ERA5,
    output=output,
)


#### Generate time series for selected ICOS stations in parallel using multiprocessing. 
The time series include ICOS measurements, ERA5 data, and downscaled ERA5 data, and are saved as CSV files.

In [ ]:
if __name__ == "__main__":
    path_icos_stat = "stations_meteoL2.csv"
    cfg = icos.load_stations_config(path_icos_stat)
    output = "out"
    os.makedirs(output, exist_ok=True)

    res = generate_timeseries_for_stations_multiprocess(
        ["DE-Geb", "CH-Dav", "FR-CLt", "IT-BCi"],
        cfg,
        output,
        start_date,
        end_date,
        "csv",
    )


Read the csv files

In [ ]:
df_ge = pd.read_csv(
    "csv/DE-Geb_timeseries.csv",
    parse_dates=["time"],
)
df_dav = pd.read_csv(
    "csv/CH-Dav_timeseries.csv",
    parse_dates=["time"],
)
df_clt = pd.read_csv(
    "csv/FR-CLt_timeseries.csv",
    parse_dates=["time"],
)


stations = {
    "Geb": df_ge,
    "Dav": df_dav,
    "CLt": df_clt,
}


Plot air and dew point temperature by the selected months for each stations

In [ ]:
plot_ta_tdp_csv(stations, "2023-04", "2023-07")

Read the csv files

In [ ]:
df_ge = pd.read_csv(
    "csv/DE-Geb_timeseries.csv",
    parse_dates=["time"],
)
df_dav = pd.read_csv(
    "csv/CH-Dav_timeseries.csv",
    parse_dates=["time"],
)
df_clt = pd.read_csv(
    "csv/FR-CLt_timeseries.csv",
    parse_dates=["time"],
)


stations = {
    "Geb": df_ge,
    "Dav": df_dav,
    "CLt": df_clt,
}


Metrics

In [ ]:
res_ge_rmse = icos.compute_monthly(
    "csv/DE-Geb_timeseries.csv",
    "rmse",  # or mae, r2, mbe or slope
)
res_ge_r2 = icos.compute_monthly(
    "csv/DE-Geb_timeseries.csv",
    "rmse",
)

res_ge_slope = icos.compute_monthly(
    "csv/DE-Geb_timeseries.csv", "slope", plot=True
)

RMSE

In [ ]:
res_ge_rmse

R2 : coefficient of determination

In [ ]:
res_ge_r2

Slope

In [ ]:
res_ge_slope

## Retrieve the time series of the observed standard atmospheric profiles and the dew point lapse rate from ICOS stations.

Download ERA5 pressure datasets date by date

In [ ]:
era5.download_date_by_date(
    date1=start_date,
    date2=end_date,
    dataset=era5.ERA5Dataset.ERA5PRESSURE,
    output=output,
)


#### Generate observed standard atmospheric and Dew Point Lapse Rates for selected ICOS stations in parallel using multiprocessing. 

In [ ]:
if __name__ == "__main__":
    path_icos_stat = "stations_meteoL2.csv"
    cfg = icos.load_stations_config(path_icos_stat)
    output = "out"
    os.makedirs(output, exist_ok=True)

    res = generate_lapse_rate_for_stations_multiprocess(
        ["DE-Geb", "CH-Dav", "FR-CLt", "IT-BCi"],
        cfg,
        output,
        start_date,
        end_date,
        "csv_lapse_rate",
    )


Read the csv files

In [ ]:
df_clt = pd.read_csv(
    "csv_lapse_rate/FR-CLt_lapse_rate.csv",
    parse_dates=["time"],
)
df_geb = pd.read_csv(
    "csv_lapse_rate/DE-Geb_lapse_rate.csv",
    parse_dates=["time"],
)
df_dav = pd.read_csv(
    "csv_lapse_rate/CH-Dav_lapse_rate.csv",
    parse_dates=["time"],
)
df_bci = pd.read_csv(
    "csv_lapse_rate/IT-BCi_lapse_rate.csv",
    parse_dates=["time"],
)

In [ ]:
stations = {"Clt": df_clt, "Geb": df_geb, "Dav": df_dav, "Bci": df_bci}

Plot the computed standart atmospehric and dew point lapse rate for each stations by the selected months

In [ ]:
plot_lr_csv(stations, "2023-11", "2024-1")